In [ ]:
!pip install unsloth anthropic pydantic plotly scipy kaleido -q
import json, os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)  # required for gated Llama base model

In [ ]:
# ── Cell 1: load test data ────────────────────────────────────────────────
import json, os, time
import numpy as np

SCORING_SYSTEM_PROMPT = """You are a Twitter content quality judge. Respond only with valid JSON.

Score the tweet on exactly these 6 dimensions, each from 1-10:

- hook_strength (weight 25%): Harry Dry 3 tests — visualizable, falsifiable, nobody else can say it. All 3 pass = 9+. Vague claims like "productivity" or "mindset" = 5 max.
- tone_compliance (weight 20%): Professional-but-direct AI practitioner voice. Six Core Laws: Zero hashtags = INSTANT 0. Zero em-dashes. Zero exclamation marks. Zero hedging. No banned words: streamline, transformative, unlock, ecosystem, landscape, game-changer.
- x_algorithm_optimization (weight 20%): X algo weights replies=27x, retweets=20x. Debate-bait + data = 9+. Any hashtag = 7 max.
- data_specificity (weight 15%): Named people, tools, numbers, falsifiable claims. Abstract = 6 max.
- pillar_alignment (weight 15%): Pillar unmistakable in first sentence. Vague opener = 6 max.
- cta_quality (weight 5%): TOFU only. Debate-bait = 9. Hard sell/link = 3 max.

Also set never_list_violation to true if any # appears in the tweet.

Return JSON:
{"hook_strength": int, "tone_compliance": int, "x_algorithm_optimization": int, "data_specificity": int, "pillar_alignment": int, "cta_quality": int, "never_list_violation": bool, "reasoning": "string max 100 words"}"""

WEIGHTS = {"hook_strength": 25, "tone_compliance": 20, "x_algorithm_optimization": 20,
           "data_specificity": 15, "pillar_alignment": 15, "cta_quality": 5}
DIMENSIONS = list(WEIGHTS.keys())

# Models sometimes output shortened key names — map them to canonical names
KEY_ALIASES = {
    "x_algorithm_optimization": ["algorithm_optimization", "x_algo_optimization",
                                 "x_algorithm", "algo_optimization"],
    "tone_compliance":          ["tone", "compliance"],
    "hook_strength":            ["hook"],
    "data_specificity":         ["data_spec", "specificity"],
    "pillar_alignment":         ["pillar", "alignment"],
    "cta_quality":              ["cta"],
}

def normalize_result(result: dict) -> dict:
    for canonical, aliases in KEY_ALIASES.items():
        if canonical not in result:
            for alias in aliases:
                if alias in result:
                    result[canonical] = result.pop(alias)
                    break
            else:
                result.setdefault(canonical, 5)  # fallback mid-range
    return result


def composite(scores: dict) -> float:
    if scores.get("never_list_violation"):
        return 0.0
    return round(sum(scores.get(d, 5) * w / 100 for d, w in WEIGHTS.items()) + 0.5, 2)


def score_with_model(model, tokenizer, content: str) -> dict:
    messages = [
        {"role": "system", "content": SCORING_SYSTEM_PROMPT},
        {"role": "user", "content": f"Score this tweet draft:\n\n{content}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    t0 = time.perf_counter()
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.0, do_sample=False)
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)
    raw = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    try:
        result = json.loads(raw)
        result = normalize_result(result)
    except json.JSONDecodeError:
        result = {d: 5 for d in DIMENSIONS}
        result.update({"never_list_violation": False, "reasoning": "parse_error"})
    result["latency_ms"] = latency_ms
    result["composite_score"] = composite(result)
    return result


test_raw = [json.loads(l) for l in open("/kaggle/input/tweet-scorer-test/test_ground_truth.jsonl")]
test_data = []
for ex in test_raw:
    s = ex["scores"]
    s["composite_score"] = composite(s)
    test_data.append({"id": ex["id"], "content": ex["content"], "haiku": s})
print(f"Loaded {len(test_data)} test examples.")

In [ ]:
# ── Cell 2: score with fine-tuned model, then free VRAM ──────────────────
import torch, gc
from unsloth import FastLanguageModel

ft_model, tokenizer = FastLanguageModel.from_pretrained(
    "sud1157/tweet-scorer-llama3-8b", max_seq_length=2048, load_in_4bit=True
)
FastLanguageModel.for_inference(ft_model)

for i, ex in enumerate(test_data):
    ex["finetuned"] = score_with_model(ft_model, tokenizer, ex["content"])
    if (i + 1) % 20 == 0:
        print(f"  finetuned: {i+1}/{len(test_data)}")

# Free VRAM before loading base model
del ft_model
gc.collect()
torch.cuda.empty_cache()
print("Fine-tuned scoring done. VRAM freed.")

In [ ]:
# ── Cell 3: score with base model, then save results ─────────────────────
base_model, _ = FastLanguageModel.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct", max_seq_length=2048, load_in_4bit=True
)
FastLanguageModel.for_inference(base_model)

for i, ex in enumerate(test_data):
    ex["base"] = score_with_model(base_model, tokenizer, ex["content"])
    if (i + 1) % 20 == 0:
        print(f"  base: {i+1}/{len(test_data)}")

del base_model
gc.collect()
torch.cuda.empty_cache()

os.makedirs("/kaggle/working/benchmark_results", exist_ok=True)
with open("/kaggle/working/benchmark_results/benchmark_results.json", "w") as f:
    json.dump(test_data, f, indent=2)
print(f"Saved {len(test_data)} results to benchmark_results.json")

In [ ]:
# ── Cell 4: compute metrics ───────────────────────────────────────────────
from scipy.stats import pearsonr

results = test_data  # already has haiku / finetuned / base

def compute_metrics(results):
    output = {}
    for model_key in ["finetuned", "base"]:
        mm = {}
        for dim in DIMENSIONS:
            h = np.array([r["haiku"][dim] for r in results])
            m = np.array([r[model_key][dim] for r in results])
            corr, _ = pearsonr(h, m)
            mm[dim] = {
                "mae": round(float(np.mean(np.abs(h - m))), 3),
                "exact_agreement": round(float(np.mean(h == m)), 3),
                "within_1_agreement": round(float(np.mean(np.abs(h - m) <= 1)), 3),
                "pearson_r": round(float(corr), 3),
            }
        hc = np.array([r["haiku"]["composite_score"] for r in results])
        mc = np.array([r[model_key]["composite_score"] for r in results])

        def tier(c): return "ready" if c >= 9.25 else ("below_target" if c >= 8.0 else "failed_floor")
        tier_acc = sum(tier(h) == tier(m) for h, m in zip(hc, mc)) / len(results)

        h_nlv = np.array([r["haiku"].get("never_list_violation", False) for r in results])
        m_nlv = np.array([r[model_key].get("never_list_violation", False) for r in results])
        tp = int(np.sum(h_nlv & m_nlv)); fp = int(np.sum(~h_nlv & m_nlv)); fn = int(np.sum(h_nlv & ~m_nlv))
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r_ = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r_ / (p + r_) if (p + r_) > 0 else 0.0

        mm["aggregate"] = {
            "mean_mae": round(float(np.mean([mm[d]["mae"] for d in DIMENSIONS])), 3),
            "mean_within_1": round(float(np.mean([mm[d]["within_1_agreement"] for d in DIMENSIONS])), 3),
            "composite_mae": round(float(np.mean(np.abs(hc - mc))), 3),
            "tier_accuracy": round(tier_acc, 3),
            "never_list_f1": round(f1, 3),
            "mean_latency_ms": round(float(np.mean([r[model_key].get("latency_ms", 0) for r in results])), 1),
        }
        output[model_key] = mm
    return output


metrics = compute_metrics(results)
print(json.dumps(metrics, indent=2))

In [ ]:
# ── generate_benchmark_card (inlined from benchmark/visualize.py) ─────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def generate_benchmark_card(metrics: dict, output_path: str) -> None:
    ft = metrics["finetuned"]
    base = metrics["base"]

    ft_mae = [ft[d]["mae"] for d in DIMENSIONS]
    base_mae = [base[d]["mae"] for d in DIMENSIONS]
    dim_labels = DIMENSIONS + [DIMENSIONS[0]]
    ft_mae_closed = ft_mae + [ft_mae[0]]
    base_mae_closed = base_mae + [base_mae[0]]

    radar_ft = go.Scatterpolar(
        r=ft_mae_closed, theta=dim_labels, fill="toself", name="Finetuned",
        line=dict(color="#1f77b4"), fillcolor="rgba(31, 119, 180, 0.15)",
    )
    radar_base = go.Scatterpolar(
        r=base_mae_closed, theta=dim_labels, fill="toself", name="Base",
        line=dict(color="#d62728"), fillcolor="rgba(214, 39, 40, 0.15)",
    )

    ft_w1 = [ft[d]["within_1_agreement"] for d in DIMENSIONS]
    base_w1 = [base[d]["within_1_agreement"] for d in DIMENSIONS]
    bar_ft_w1 = go.Bar(x=DIMENSIONS, y=ft_w1, name="Finetuned", marker_color="#1f77b4", showlegend=False)
    bar_base_w1 = go.Bar(x=DIMENSIONS, y=base_w1, name="Base", marker_color="#d62728", showlegend=False)

    agg_labels = ["Composite MAE", "Tier Accuracy", "Never-list F1", "Mean Latency (ms)"]
    agg_keys = ["composite_mae", "tier_accuracy", "never_list_f1", "mean_latency_ms"]
    ft_agg = [ft["aggregate"][k] for k in agg_keys]
    base_agg = [base["aggregate"][k] for k in agg_keys]
    bar_ft_agg = go.Bar(x=agg_labels, y=ft_agg, name="Finetuned", marker_color="#1f77b4", showlegend=False)
    bar_base_agg = go.Bar(x=agg_labels, y=base_agg, name="Base", marker_color="#d62728", showlegend=False)

    cost_table = go.Table(
        header=dict(
            values=["<b>Option</b>", "<b>Cost / 1K calls</b>", "<b>Approx latency</b>"],
            fill_color="#2c3e50", font=dict(color="white", size=12), align="left",
        ),
        cells=dict(
            values=[
                ["Haiku API (claude-haiku-3)", "Self-hosted (A10G / vLLM)"],
                ["$0.50", "~$0.00"],
                ["~800 ms", "~400 ms"],
            ],
            fill_color=[["#ecf0f1", "#dfe6e9"]], align="left", font=dict(size=12),
        ),
    )

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "MAE per Dimension (lower = better)",
            "Within-1 Agreement per Dimension",
            "Aggregate Metrics",
            "Cost Comparison",
        ),
        specs=[[{"type": "polar"}, {"type": "xy"}], [{"type": "xy"}, {"type": "table"}]],
        vertical_spacing=0.18,
        horizontal_spacing=0.12,
    )
    fig.add_trace(radar_ft, row=1, col=1)
    fig.add_trace(radar_base, row=1, col=1)
    fig.add_trace(bar_ft_w1, row=1, col=2)
    fig.add_trace(bar_base_w1, row=1, col=2)
    fig.add_trace(bar_ft_agg, row=2, col=1)
    fig.add_trace(bar_base_agg, row=2, col=1)
    fig.add_trace(cost_table, row=2, col=2)

    fig.update_layout(
        title=dict(text="Tweet-Scorer Fine-Tune Benchmark Card", font=dict(size=20), x=0.5, xanchor="center"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        barmode="group", height=900, width=1400, template="plotly_white",
        margin=dict(t=100, b=60, l=60, r=60),
    )
    fig.update_polars(radialaxis=dict(autorange="reversed"))
    fig.update_yaxes(range=[0, 1], row=1, col=2, title_text="Agreement Rate")
    fig.update_xaxes(tickangle=-30, row=1, col=2)
    fig.update_xaxes(tickangle=-20, row=2, col=1)
    fig.update_yaxes(title_text="Value", row=2, col=1)

    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    fig.write_html(output_path + ".html")
    fig.write_image(output_path + ".png")  # requires kaleido


generate_benchmark_card(metrics, "/kaggle/working/benchmark_card")
print("Download benchmark_card.html and benchmark_card.png from /kaggle/working/")